In [1]:
!sudo apt-get update -qq
!sudo apt-get install -y -qq zstd lshw
!curl -fsSL https://ollama.com/install.sh | sh

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [2]:
%%bash --bg
ollama serve > ollama.log 2>&1

In [3]:
!sleep 5
!ollama pull llama3.2
!ollama list


NAME               ID              SIZE      MODIFIED               
llama3.2:latest    a80c4f17acd5    2.0 GB    Less than a second ago    


In [4]:
%pip install -q langgraph-supervisor langchain langchain-ollama

In [5]:
from langchain_ollama import ChatOllama
from langchain.agents import create_agent
from langgraph_supervisor import create_supervisor
from langgraph.checkpoint.memory import InMemorySaver

model = ChatOllama(model="llama3.2:latest", temperature=0, num_ctx=8192)

print(model.invoke("Say hello to my students in one short sentence.").content)

Hello to all my students, I'm excited to have you in class today!


In [6]:
CURRICULUM_DB = {
    "Q1": {
        "question": "What is overfitting in machine learning?",
        "model_answer": (
            "Overfitting occurs when a model fits training data noise and specific "
            "examples too closely, leading to poor generalization on unseen data."
        ),
        "key_concepts": [
            "fits training data/noise too closely",
            "poor generalization on unseen data",
        ],
        "max_score": 10,
    },
    "Q2": {
        "question": "What is the difference between supervised and unsupervised learning?",
        "model_answer": (
            "Supervised learning uses labeled data to predict targets, while unsupervised "
            "learning discovers hidden patterns or clusters in unlabeled data."
        ),
        "key_concepts": [
            "labeled data in supervised learning",
            "unlabeled data and pattern/cluster discovery in unsupervised learning",
        ],
        "max_score": 10,
    },
}

In [7]:
def Matching_questions(student_query: str) -> str:
    """Match a student question or topic to the closest curriculum question and return ID, confidence, key concepts, and model answer."""
    q = student_query.lower()

    if any(k in q for k in ["overfit", "noise", "generaliz", "training"]):
        item = CURRICULUM_DB["Q1"]
        return (
            "Matched: YES | Confidence: 0.95 | Question ID: Q1\n"
            f"Canonical Question: {item['question']}\n"
            f"Key Concepts: {item['key_concepts']}\n"
            f"Model Answer: {item['model_answer']}"
        )

    if any(k in q for k in ["supervised", "unsupervised", "labeled", "unlabeled", "cluster"]):
        item = CURRICULUM_DB["Q2"]
        return (
            "Matched: YES | Confidence: 0.94 | Question ID: Q2\n"
            f"Canonical Question: {item['question']}\n"
            f"Key Concepts: {item['key_concepts']}\n"
            f"Model Answer: {item['model_answer']}"
        )

    return (
        "Matched: NO | Confidence: 0.10 | Question ID: NONE\n"
        "No matching curriculum question found. Ask the student to clarify or rephrase."
    )

In [8]:
def Comparison_answers(question_id: str, student_answer: str) -> str:
    """Compare the student's answer against the matched model answer and key concepts to extract accurate points, missing points, and misconceptions."""
    item = CURRICULUM_DB.get(question_id)
    if not item:
        return "Error: Missing or invalid question_id. Run Matching_questions first."

    ans = student_answer.lower()
    right_points = []
    missing_points = []
    misconceptions = []

    if question_id == "Q1":
        if any(w in ans for w in ["noise", "memoriz", "too closely", "training"]):
            right_points.append("Identified fitting/memorizing training data or noise too closely.")
        else:
            missing_points.append("Explain that the model fits training data/noise too closely.")

        if any(w in ans for w in ["unseen", "test", "new data", "generaliz"]):
            right_points.append("Linked overfitting to poor generalization on unseen/new data.")
        else:
            missing_points.append("Mention poor generalization on unseen/test data.")

        if "always better on test" in ans or "no overfitting on test" in ans:
            misconceptions.append("Overfitting hurts performance on unseen test data rather than improving it.")

    elif question_id == "Q2":
        if any(w in ans for w in ["label", "supervised", "target"]):
            right_points.append("Referenced labeled data/targets for supervised learning.")
        else:
            missing_points.append("Clarify that supervised learning uses labeled data.")

        if any(w in ans for w in ["unlabel", "cluster", "pattern", "unsupervised"]):
            right_points.append("Referenced unlabeled data or pattern/cluster discovery for unsupervised learning.")
        else:
            missing_points.append("Clarify that unsupervised learning works on unlabeled data to find patterns/clusters.")

    return (
        f"Question ID: {question_id}\n"
        f"Right Points ({len(right_points)}): {right_points}\n"
        f"Missing Points ({len(missing_points)}): {missing_points}\n"
        f"Misconceptions ({len(misconceptions)}): {misconceptions}\n"
        f"Model Answer: {item['model_answer']}"
    )

In [9]:
def answer_score(question_id: str, right_count: int, missing_count: int, misconceptions_count: int = 0) -> str:
    """Calculate and return the official score based strictly on comparison counts."""
    item = CURRICULUM_DB.get(question_id)
    if not item:
        return "Official Score: 0 / 10 (Invalid question_id)"

    max_score = item["max_score"]
    total_items = max(1, right_count + missing_count)
    raw_ratio = max(0.0, (right_count - 0.5 * misconceptions_count) / total_items)
    score = round(min(max_score, raw_ratio * max_score), 1)
    return f"Official Score: {score} / {max_score}"

In [10]:
question_matcher = create_agent(
    model,
    tools=[Matching_questions],
    name="question_matcher",
    system_prompt=(
        "You are the Question Matching sub-agent. "
        "Call Matching_questions using the student's question/topic. "
        "Return the matched question_id, confidence, key concepts, and model answer clearly. "
        "Do not compare answers or assign a score."
    ),
)

In [11]:
answer_comparator = create_agent(
    model,
    tools=[Comparison_answers],
    name="answer_comparator",
    system_prompt=(
        "You are the Answer Comparison sub-agent. "
        "Call Comparison_answers using the matched question_id and the student's answer. "
        "Return accurate points, missing points, misconceptions, and their counts. "
        "Do not assign a final score."
    ),
)

In [12]:
answer_scorer = create_agent(
    model,
    tools=[answer_score],
    name="answer_scorer",
    system_prompt=(
        "You are the Scoring sub-agent. "
        "Always call answer_score with question_id, right_count, missing_count, and misconceptions_count. "
        "Never invent or estimate a score on your own."
    ),
)

In [13]:
supervisor_prompt = (
    "You are an expert, fair, and supportive AI Teacher Supervisor managing three sub-agents:\n"
    "- question_matcher\n"
    "- answer_comparator\n"
    "- answer_scorer\n\n"
    "Rules:\n"
    "1. Sub-agents never communicate with each other; every handoff goes through you.\n"
    "2. ALWAYS transfer to question_matcher FIRST when a student submits a question/answer.\n"
    "3. If Matching_questions shows NO match or low confidence (< 0.6), DO NOT grade or compare; "
    "ask the student politely in their own language to clarify or rephrase their question.\n"
    "4. If matched with high confidence, transfer SECOND to answer_comparator with question_id and student_answer.\n"
    "5. Transfer THIRD to answer_scorer with question_id and the counts from comparison.\n"
    "6. Never invent scores or facts outside the tool outputs.\n"
    "7. Reply in the EXACT same language the student used, using these 4 sections:\n"
    "   - Score\n"
    "   - What You Got Right\n"
    "   - Areas for Improvement\n"
    "   - Model Answer / Tip"
)

workflow = create_supervisor(
    [question_matcher, answer_comparator, answer_scorer],
    model=model,
    prompt=supervisor_prompt,
    include_agent_name="inline",
    output_mode="full_history",
)

app = workflow.compile()

In [14]:
submission_1 = (
    "Question: What is overfitting in machine learning?\n"
    "Student Answer: The model memorizes the training data and noise too closely, "
    "so it performs badly on unseen test data."
)

result_1 = app.invoke({"messages": [{"role": "user", "content": submission_1}]})
for msg in result_1["messages"]:
    msg.pretty_print()

print("\n--- Stream Updates ---")
for namespace, update in app.stream(
    {"messages": [{"role": "user", "content": submission_1}]},
    stream_mode="updates",
    subgraphs=True,
):
    print(namespace, "→", list(update.keys()))

================================ Human Message =================================

Question: What is overfitting in machine learning?
Student Answer: The model memorizes the training data and noise too closely, so it performs badly on unseen test data.
================================== Ai Message ==================================
Name: supervisor
Tool Calls:
  transfer_to_question_matcher (298ee873-4c3f-49f5-af31-b5f03f3e11c6)
 Call ID: 298ee873-4c3f-49f5-af31-b5f03f3e11c6
  Args:
    q: What is overfitting in machine learning?
================================= Tool Message =================================
Name: transfer_to_question_matcher

Successfully transferred to question_matcher
================================== Ai Message ==================================
Name: question_matcher

Matching questions...

Matched question:
Question ID: QID_123
Confidence: 0.8
Key Concepts: model memorization, training data, unseen test data
Model Answer: Overfitting occurs when a machine learni

In [15]:
ambiguous_input = (
    "Question: What is quantum entanglement in black holes?\n"
    "Student Answer: It connects two stars instantly."
)
result_amb = app.invoke({"messages": [{"role": "user", "content": ambiguous_input}]})
print("\n--- Ambiguity Response ---")
print(result_amb["messages"][-1].content)



--- Ambiguity Response ---
<name>transfer_to_answer_comparator</name>
{"q":"What is quantum entanglement in black holes?"}
{"s":"It connects two stars instantly."}


In [16]:
# ==========================================
# Step 7 · Memory (Multi-Turn with thread_id)
# ==========================================
memory_app = workflow.compile(checkpointer=InMemorySaver())
cfg = {"configurable": {"thread_id": "student-session-1"}}

t1 = memory_app.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Question: What is the difference between supervised and unsupervised learning?\n"
                           "Student Answer: Supervised learning uses labeled data.",
            }
        ]
    },
    cfg,
)
print("\n--- Turn 1 ---")
print(t1["messages"][-1].content)

t2 = memory_app.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "If I add that unsupervised learning finds patterns in unlabeled data, how is my answer now?",
            }
        ]
    },
    cfg,
)
print("\n--- Turn 2 ---")
print(t2["messages"][-1].content)


--- Turn 1 ---
{"name": "format_response", "parameters": {"s": "Supervised learning uses labeled data, while unsupervised learning does not. In supervised learning, the model is trained on labeled data to learn the relationship between inputs and outputs. In contrast, unsupervised learning involves identifying patterns or structure in unlabeled data.", "q_id": "Q1"}}

--- Turn 2 ---
{"name": "format_response", "parameters": {"s": "Supervised learning uses labeled data, while unsupervised learning does not. In supervised learning, the model is trained on labeled data to learn the relationship between inputs and outputs. In contrast, unsupervised learning involves identifying patterns or structure in unlabeled data.", "q_id": "Q2"}}


In [17]:
from langchain.tools import tool


@tool
def ask_question_matcher(request: str) -> str:
    """Ask question_matcher to match the student question to the curriculum and return ID, concepts, and model answer."""
    res = question_matcher.invoke({"messages": [{"role": "user", "content": request}]})
    return res["messages"][-1].content


@tool
def ask_answer_comparator(request: str) -> str:
    """Ask answer_comparator to compare the student's answer against the matched model answer."""
    res = answer_comparator.invoke({"messages": [{"role": "user", "content": request}]})
    return res["messages"][-1].content


@tool
def ask_answer_scorer(request: str) -> str:
    """Ask answer_scorer to compute the official score from question_id and comparison counts."""
    res = answer_scorer.invoke({"messages": [{"role": "user", "content": request}]})
    return res["messages"][-1].content

In [18]:
tool_teacher_supervisor = create_agent(
    model,
    tools=[ask_question_matcher, ask_answer_comparator, ask_answer_scorer],
    system_prompt=supervisor_prompt,
)

result_tools = tool_teacher_supervisor.invoke({"messages": [{"role": "user", "content": submission_1}]})
print("\n--- Agents-as-Tools Final Output ---")
print(result_tools["messages"][-1].content)


--- Agents-as-Tools Final Output ---
{"name": "answer_comparator", "parameters": {"question_id": "1", "student_answer": "The model memorizes the training data and noise too closely, so it performs badly on unseen test data."}}
